# DRAGNET - robustness of non-monotonicity & disjointness to greedy near-ties (Kaggle)

Answers reviewer R1.1: is "adding a passage breaks reproduction" real content-redirection or just a
logit near-tie flipping the argmax? Per violating add S -> S u {p}, this logs (a) the drop in the wrong
answer's log-probability (near-tie = small drop) and (b) re-verifies the flip under temperature sampling
(counts it only if S still reproduces and S u {p} still breaks on a majority of samples). Reports the
non-monotone and disjoint rates greedy vs resampled vs high-margin, so you can see how much survives.

**Settings:** GPU **T4** (4-bit, k=6 -> fast) - Internet **on**. ~30-60 min at 120 cases.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/kaggle/working'); os.chdir(WORK)
def fetch(n,u):
    if (WORK/n).exists(): subprocess.run(['git','-C',n,'pull','--ff-only'],check=True)
    else: subprocess.run(['git','clone','--depth','1',u,n],check=True)
fetch('lineup','https://github.com/santoshcheethiralame-dot/LINEUP')
ds=next((p.parent for p in Path('/kaggle/input').glob('*/**/src/dragnet/__init__.py')),None)
if ds is not None:
    shutil.rmtree(WORK/'dragnet',ignore_errors=True); shutil.copytree(ds.parent.parent,WORK/'dragnet')
else: fetch('dragnet','https://github.com/santoshcheethiralame-dot/DRAGNET')
subprocess.run([sys.executable,'-m','pip','install','-q','-e','./lineup[gpu]'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e','./dragnet'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','bitsandbytes>=0.46.1'],check=True)
for pkg in ('lineup','dragnet'):
    s=str(WORK/pkg/'src')
    if s not in sys.path: sys.path.insert(0,s)
print('[ok] ready')

In [ ]:
MODEL='Qwen/Qwen2.5-7B-Instruct'; TAG='qwen'; DATASET='hotpotqa'
LIMIT_BUILD=300      # source questions for the natural cell
LIMIT_CASES=120      # wrong cases to probe
MAX_SIZE=3           # enumeration bound for minimal sufficient sets
N_SAMPLES=5          # temperature samples per subset
TEMPERATURE=0.7
SEED=0
cell=f'runs/{DATASET}/natural/{TAG}'
print(cell)

In [ ]:
# Build the natural (unplanted) k=6 cell if not already present, then run the robustness probe.
import subprocess, sys
from pathlib import Path
if not (Path(cell)/'generations.jsonl').exists():
    print('==== building natural cell ====', flush=True)
    subprocess.run([sys.executable,'lineup/scripts/run_pipeline.py','--dataset',DATASET,'--natural',
        '--limit',str(LIMIT_BUILD),'--k','6','--seed',str(SEED),'--model',MODEL,'--load-in-4bit',
        '--out',cell],check=True)
print('==== robust probe ====', flush=True)
subprocess.run([sys.executable,'dragnet/scripts/run_robust_probe.py','--cell',cell,'--model',MODEL,
    '--load-in-4bit','--limit',str(LIMIT_CASES),'--max-size',str(MAX_SIZE),
    '--n-samples',str(N_SAMPLES),'--temperature',str(TEMPERATURE),'--seed',str(SEED)],check=True)

In [ ]:
import shutil
stem=f'dragnet_{DATASET}_{TAG}_robust_seed{SEED}'
shutil.make_archive(stem,'zip','runs')
print('download:',f'/kaggle/working/{stem}.zip')